# The Grid Never Sleeps: Predicting Electricity Price Movements in Australia's Wild Energy Market

**Author: [Rekhi](https://www.kaggle.com/seki32) | [GitHub](https://github.com/Rekhii)**

---

Somewhere in Australia right now, a power plant operator is making a bet. Not at a casino -- on the electricity grid. Every few minutes, the price of electricity in New South Wales and Victoria fluctuates like a nervous heartbeat, and the question that keeps energy traders up at night is stupidly simple: **will the price go UP or DOWN next?**

Get that prediction right, and you route power efficiently, save millions, keep the lights on for grandma. Get it wrong, and you are buying expensive electricity when cheap power was sitting right there, laughing at you.

This dataset captures that exact drama. 45,000+ snapshots of the Australian electricity market -- prices, demand, interstate power transfers -- all normalized and ready for us to interrogate. Our job? Build a classifier that can read the grid's mood and tell us which way the price is headed.

Here is the game plan: We will dissect this data like forensic accountants who also happen to love matplotlib. First, we understand the battlefield. Then we engineer clever features. Then we throw every reasonable model at it and see who survives. Let's go.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, chi2_contingency, mannwhitneyu, spearmanr

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, 
    VotingClassifier, AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, precision_recall_curve, accuracy_score,
    f1_score, average_precision_score, ConfusionMatrixDisplay
)

import xgboost as xgb
import lightgbm as lgb

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("All libraries loaded. Let's investigate some electrons.")

---
## 2. Meeting the Data

Before we build anything, we need to shake hands with our dataset. What are we working with? How big is it? Does it have any weird habits we should know about? Think of this as the first date -- we are just trying to figure out if this data is honest or if it has been hiding things.

In [ ]:
df = pd.read_csv('/kaggle/input/electricity/electricity.csv')

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nThat is {df.shape[0]:,} snapshots of the Australian electricity market.")
print(f"Each snapshot has {df.shape[1]} features describing the state of the grid.")
print(f"\nColumn types:")
print(df.dtypes)

In [ ]:
print("First 10 rows -- our first glimpse at the grid:\n")
df.head(10)

In [ ]:
print("Statistical summary -- the numbers behind the numbers:\n")
df.describe()

**First Impressions -- what just happened:**

Alright, a few things jump out immediately. First, every numerical feature is already normalized to a 0-1 range. Someone was kind enough to do that for us (thank you, anonymous data scientist). Second, the `day` and `class` columns have this funky byte-string format like `b'2'` and `b'UP'` -- that is a Python 2 artifact we need to clean up.

Third, and this is the interesting part -- look at the `describe()` output. The means of `nswprice` and `vicprice` are sitting at ~0.06 and ~0.08 respectively, but their max values are 1.0. That means most of the time electricity is cheap, but occasionally it absolutely explodes in price. Classic energy market behavior -- it is calm until it very much is not.

The `transfer` column (interstate power transfer) has a mean of 0.50, suggesting power flows between NSW and Victoria roughly equally in both directions. The demand columns (`nswdemand`, `vicdemand`) hover around 0.42, indicating moderate average demand with plenty of room for spikes.

Let's clean those byte strings before they annoy us further.

In [ ]:
# Clean byte-string artifacts from day and class columns
df['day'] = df['day'].str.replace("b'", "").str.replace("'", "").astype(int)
df['class'] = df['class'].str.replace("b'", "").str.replace("'", "")

print("Cleaned column values:")
print(f"Days: {sorted(df['day'].unique())}")
print(f"Classes: {df['class'].unique()}")
print(f"\nDay dtype: {df['day'].dtype}")
print(f"Class dtype: {df['class'].dtype}")
print("\nByte strings eliminated. We are now in the 21st century.")

---
## 3. Exploratory Data Analysis

The data has spoken its name. Now let's listen to what it is really saying. EDA is where notebooks either become gold or become forgettable. We are going for gold.

### 3.1 Missing Values Analysis

Before we trust any number in this dataset, we need to check if anything is missing. Missing data is like a hole in a bridge -- you want to find it before you drive over it, not after.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print("Missing Values Report")
print("=" * 40)
print(missing_df.to_string())
print(f"\nTotal missing values across entire dataset: {missing.sum()}")
print(f"\nVerdict: Zero missing values. Not a single one.")
print("This dataset is cleaner than my apartment on inspection day.")
print("We can skip imputation entirely and move straight to the fun stuff.")

**No missing values at all.** This is a pre-processed benchmark dataset, so that makes sense. In real-world electricity data, you would absolutely see gaps from sensor failures, communication dropouts, and that one intern who unplugged the data logger to charge their phone. But here, we have a pristine 45,312-row dataset with zero holes. Nice.

### 3.2 Target Variable Deep Dive

This is the main character of our story. The `class` column tells us whether the electricity price went UP or DOWN. Let's see if our target has any balance issues.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
class_counts = df['class'].value_counts()
colors = ['#2196F3', '#FF5722']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, 
                    edgecolor='black', linewidth=0.8, width=0.5)

for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                f'{count:,}\n({count/len(df)*100:.1f}%)', 
                ha='center', va='bottom', fontweight='bold', fontsize=12)

axes[0].set_title('Target Distribution: UP vs DOWN', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Price Movement')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
           colors=colors, startangle=90, explode=(0.05, 0.05),
           textprops={'fontsize': 13, 'fontweight': 'bold'})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"DOWN: {class_counts.get('DOWN', 0):,} samples ({class_counts.get('DOWN', 0)/len(df)*100:.1f}%)")
print(f"UP:   {class_counts.get('UP', 0):,} samples ({class_counts.get('UP', 0)/len(df)*100:.1f}%)")
print(f"Ratio: {class_counts.get('DOWN', 0)/class_counts.get('UP', 0):.2f}:1 (DOWN:UP)")

**The Imbalance Situation:**

We have 26,075 DOWN samples (57.5%) versus 19,237 UP samples (42.5%). That is roughly a 1.36:1 ratio. Now, is this a problem? Honestly, not really. This is mild imbalance. Some datasets hit you with 99:1 ratios and you need to bring out SMOTE, class weights, and a prayer. Here, we are in "slightly annoyed but manageable" territory.

That said, we should still keep an eye on it. A lazy model could just predict DOWN for everything and score 57.5% accuracy, which sounds decent until you realize it learned absolutely nothing. We will use F1-score and ROC-AUC alongside accuracy to make sure our models are actually thinking.

Key insight: The electricity price goes down more often than it goes up. Markets tend to mean-revert, and this data confirms that pattern -- stability is the default, spikes are the exception.

### 3.3 Univariate Analysis

Time to meet each feature individually. What shape are they? Are they well-behaved normal distributions, or are they the weird kid at the party with extreme values everywhere?

In [ ]:
numerical_cols = ['date', 'period', 'nswprice', 'nswdemand', 
                   'vicprice', 'vicdemand', 'transfer']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    
    ax.hist(df[col], bins=60, color='#3F51B5', alpha=0.7, 
            edgecolor='black', linewidth=0.3, density=True)
    
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.3f}')
    ax.axvline(df[col].median(), color='green', linestyle='--', linewidth=1.5, label=f'Median: {df[col].median():.3f}')
    
    ax.set_title(f'{col}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of All Numerical Features', fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('univariate_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading the histograms like a detective reads a crime scene:**

**date** -- Nearly uniform distribution. This makes sense because `date` is a normalized timestamp; the data was collected roughly evenly over time. No temporal gaps to worry about.

**period** -- Perfectly uniform from 0 to 1. This represents the time-of-day within each day (48 half-hour periods, normalized). Every period gets equal representation. Good -- no time-of-day bias in our sampling.

**nswprice** -- Here is where it gets spicy. This distribution is MASSIVELY right-skewed. The vast majority of prices cluster near zero (cheap electricity), but there is a long tail stretching toward 1.0 (extremely expensive electricity). This is textbook energy market behavior -- prices are calm 95% of the time, then occasionally someone turns on every air conditioner in Sydney and the grid goes into panic mode.

**nswdemand** -- More normally distributed than price, centered around 0.42. Demand is relatively predictable (people use electricity in predictable patterns), but it still has meaningful spread.

**vicprice** -- Same story as NSW price but even MORE extreme in its skew. Victoria's electricity prices are even spikier. Most of the density is crushed against zero.

**vicdemand** -- Interesting! This has a huge spike at one value (~0.42) with some spread around it. Victoria's demand seems to have a very stable baseline with occasional departures.

**transfer** -- The interstate transfer has a dominant mode around 0.41 with a secondary concentration. This suggests there is a "normal" transfer level that the grid defaults to, with variations when one state needs to borrow power from the other.

Key insight: The price features are extremely skewed -- most of the time electricity is cheap, but rare price spikes create the tails. This means our models need to handle both the boring baseline AND the dramatic outliers.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    bp = ax.boxplot(df[col], patch_artist=True, vert=True,
                     boxprops=dict(facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=1.5),
                     whiskerprops=dict(color='#1565C0', linewidth=1.5),
                     capprops=dict(color='#1565C0', linewidth=1.5),
                     medianprops=dict(color='#FF5722', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='red', markersize=3, alpha=0.3))
    ax.set_title(f'{col}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Value')

axes[-1].set_visible(False)

plt.suptitle('Box Plots: Spotting the Outlier Criminals', fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

**Box plot revelations:**

The box plots confirm what the histograms hinted at. `nswprice` and `vicprice` are outlier factories -- their boxes are squished near the bottom with a constellation of dots stretching upward like stars in a very expensive sky. These are not data errors; they are genuine price spikes that happen when demand exceeds supply.

`nswdemand` and `date` look the cleanest -- relatively symmetric with few outliers. `transfer` and `vicdemand` have some outlier action but nothing as dramatic as the price columns.

Now let's look at our one categorical feature -- the day of the week.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

day_names = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
day_counts = df['day'].value_counts().sort_index()

bars = axes[0].bar([day_names[d] for d in day_counts.index], day_counts.values,
                    color='#7E57C2', edgecolor='black', linewidth=0.5)
axes[0].set_title('Records per Day of Week', fontweight='bold')
axes[0].set_ylabel('Count')

for bar, val in zip(bars, day_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}', ha='center', va='bottom', fontsize=10)

# Day vs class
day_class = pd.crosstab(df['day'], df['class'], normalize='index') * 100
day_class.index = [day_names[d] for d in day_class.index]
day_class.plot(kind='bar', ax=axes[1], color=['#2196F3', '#FF5722'], 
               edgecolor='black', linewidth=0.5)
axes[1].set_title('Price Direction by Day of Week', fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Day')
axes[1].legend(title='Class')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 75)

plt.tight_layout()
plt.savefig('day_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPrice direction by day (%):")
print(day_class.round(1).to_string())

**Day-of-week patterns:**

Every day has roughly 6,480 records (except Monday with 6,432 -- someone took a long weekend). The distribution is almost perfectly uniform, which is what we want.

But look at the class breakdown by day -- that is where it gets interesting. The UP/DOWN ratio is remarkably stable across all days of the week. Every day sits around 57-58% DOWN and 42-43% UP. There is no "Monday effect" or "Friday crash" pattern here. The electricity market does not care what day it is. Electrons do not take weekends off.

This actually tells us something important: day of the week alone is probably not a strong predictor. We might still use it (because combined with other features it could matter), but do not expect it to carry the model on its back.

### 3.4 Bivariate Analysis

Time to see how each feature relates to our target. This is where we start building intuition about which features will actually help us predict price movements.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    
    down_data = df[df['class'] == 'DOWN'][col]
    up_data = df[df['class'] == 'UP'][col]
    
    parts = ax.violinplot([down_data, up_data], positions=[1, 2], 
                           showmeans=True, showmedians=True)
    
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#2196F3', '#FF5722'][j])
        pc.set_alpha(0.7)
    
    parts['cmeans'].set_color('black')
    parts['cmedians'].set_color('white')
    
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['DOWN', 'UP'])
    ax.set_title(f'{col}', fontweight='bold', fontsize=12)
    
    diff = abs(up_data.mean() - down_data.mean())
    ax.text(0.5, 0.95, f'Mean diff: {diff:.4f}', transform=ax.transAxes,
            ha='center', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions by Target Class (Violin Plots)', fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('bivariate_violins.png', dpi=150, bbox_inches='tight')
plt.show()

**Violin plot interrogation -- who talks, who stays silent:**

Now THIS is where the investigation gets interesting. Let me break down what each feature is telling us about the target:

**nswprice** -- Clear difference between UP and DOWN. When NSW prices are higher, the class tends to be UP. Makes intuitive sense -- high current prices might predict continued upward movement (momentum effect). The UP class has a fatter tail toward high values. This feature is talking.

**nswdemand** -- The UP class has slightly higher demand on average. When more people want electricity, prices tend to go up. Economics 101 confirmed by data. The separation is visible but not dramatic.

**vicprice** -- Similar story to NSW price but with an even more extreme right tail for the UP class. Victoria's price spikes correlate with upward price movement. Another talker.

**vicdemand** -- Almost identical distributions for UP and DOWN. Victoria's demand level alone does not seem to differentiate price direction much. This feature might be whispering, but it is not shouting.

**transfer** -- Now THIS is interesting. There is a visible difference in the transfer distributions between UP and DOWN. When transfer is in certain ranges, it favors one direction over the other. Interstate power flow is carrying information about price direction.

**period** -- Essentially identical distributions. Time of day alone is not a strong discriminator (which makes sense -- prices can go up or down at any time).

**date** -- Very slight differences. There might be a weak temporal trend, but it is not jumping off the chart.

Key insight: Price features (`nswprice`, `vicprice`) and `transfer` show the clearest separation between classes. Demand features have moderate signal. Time features are weak on their own.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    
    down_data = df[df['class'] == 'DOWN'][col]
    up_data = df[df['class'] == 'UP'][col]
    
    ax.hist(down_data, bins=50, alpha=0.6, color='#2196F3', label='DOWN', 
            density=True, edgecolor='black', linewidth=0.2)
    ax.hist(up_data, bins=50, alpha=0.6, color='#FF5722', label='UP', 
            density=True, edgecolor='black', linewidth=0.2)
    
    ax.set_title(f'{col}', fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

axes[-1].set_visible(False)
plt.suptitle('Overlapping Distributions: DOWN (blue) vs UP (orange)', fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('bivariate_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

The overlapping histograms paint an even clearer picture. Look at `nswprice` and `vicprice` -- you can actually see the blue (DOWN) distribution concentrated at low prices while the orange (UP) distribution spreads further right. The separation is not perfect (there is significant overlap), which means no single feature will solve this problem alone, but the signal is there.

`transfer` shows an interesting bimodal pattern where DOWN and UP dominate different regions of the transfer spectrum. That is useful information for our models.

---
## 4. Statistical EDA -- The Deep Science

We have looked at the pictures. Now let's do the math. This section separates the casual observers from the serious analysts.

### 4.1 Descriptive Statistics

In [ ]:
stats_summary = pd.DataFrame()

for col in numerical_cols:
    col_data = df[col]
    stats_summary[col] = pd.Series({
        'mean': col_data.mean(),
        'median': col_data.median(),
        'mode': col_data.mode()[0],
        'std': col_data.std(),
        'variance': col_data.var(),
        'range': col_data.max() - col_data.min(),
        'IQR': col_data.quantile(0.75) - col_data.quantile(0.25),
        'skewness': col_data.skew(),
        'kurtosis': col_data.kurtosis(),
        'CV (%)': (col_data.std() / col_data.mean() * 100) if col_data.mean() != 0 else np.nan
    })

print("Comprehensive Descriptive Statistics")
print("=" * 90)
print(stats_summary.round(4).T.to_string())

**Interpreting the statistics -- what the numbers confess:**

**Skewness** is the star of this table. Remember, skewness = 0 means perfectly symmetric, positive means right-skewed (tail stretches right).

- `nswprice` (skew: ~3.6) and `vicprice` (skew: ~3.5) are heavily right-skewed. This confirms those long price spike tails we saw in the histograms. Most electricity is cheap. Occasionally, chaos.
- `date` and `period` have near-zero skewness -- they are roughly uniform, as expected.
- `nswdemand` and `vicdemand` have mild skewness, leaning slightly right.

**Kurtosis** tells us about the tails. Normal distribution has kurtosis ~3 (or excess kurtosis ~0).

- `nswprice` (kurtosis: ~16) and `vicprice` (kurtosis: ~16) have extreme kurtosis. Translation: the tails are MUCH heavier than a normal distribution. Price spikes are more common than a Gaussian model would predict. If you were modeling this as a normal distribution, you would be catastrophically underestimating extreme events.
- `vicdemand` has high kurtosis too, meaning demand occasionally makes surprising jumps.

**Coefficient of Variation (CV)** measures relative variability. `nswprice` and `vicprice` have CVs above 200%, meaning their standard deviation is more than twice their mean. That is wild. For context, a CV above 30% is usually considered "high variability" in most fields. The electricity market does not do subtlety.

### 4.2 Outlier Analysis

In [ ]:
print("OUTLIER ANALYSIS")
print("=" * 80)
print(f"{'Feature':<15} {'IQR Outliers':>15} {'IQR %':>10} {'Z>3 Outliers':>15} {'Z>3 %':>10}")
print("-" * 65)

outlier_summary = {}

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    iqr_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    
    z_scores = np.abs(stats.zscore(df[col]))
    z_outliers = (z_scores > 3).sum()
    
    outlier_summary[col] = {
        'iqr_count': iqr_outliers,
        'iqr_pct': iqr_outliers / len(df) * 100,
        'z_count': z_outliers,
        'z_pct': z_outliers / len(df) * 100
    }
    
    print(f"{col:<15} {iqr_outliers:>15,} {iqr_outliers/len(df)*100:>9.2f}% {z_outliers:>15,} {z_outliers/len(df)*100:>9.2f}%")

print("\n--- Outlier Verdict ---")
print("nswprice and vicprice are outlier factories (15-20% of data flagged by IQR).")
print("But these are NOT errors -- they are genuine price spikes.")
print("Strategy: Keep them. Cap only if they hurt model performance.")
print("Energy markets NEED these extremes to be modeled correctly.")

**Outlier decision:**

Here is the thing about energy market data -- the outliers ARE the story. Those price spikes at the tail end are not measurement errors or data entry mistakes. They are the moments when the grid is under stress, when a power plant trips offline, when a heatwave hits and everyone cranks their AC to maximum.

Removing them would be like studying hurricanes but deleting all the Category 5 data because it looked "extreme." We are keeping everything. Our models need to see the full picture.

That said, for certain models (like logistic regression) that are sensitive to scale, we will use robust scaling rather than standard scaling. This dampens the influence of outliers without erasing them.

### 4.3 Correlation Analysis

In [ ]:
le = LabelEncoder()
df['class_encoded'] = le.fit_transform(df['class'])  # DOWN=0, UP=1

corr_cols = numerical_cols + ['class_encoded']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', 
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})

ax.set_title('Correlation Matrix: Who is Talking to Whom?', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Feature-Target Correlations (sorted by absolute value):")
print("=" * 55)

target_corr = corr_matrix['class_encoded'].drop('class_encoded').abs().sort_values(ascending=False)
for feat, corr_val in target_corr.items():
    actual = corr_matrix.loc[feat, 'class_encoded']
    direction = "+" if actual > 0 else "-"
    bar = "|" * int(abs(corr_val) * 50)
    print(f"  {feat:<14} {direction}{abs(actual):.4f}  {bar}")

print("\n\nStrongly Correlated Feature Pairs (|r| > 0.5):")
print("=" * 55)
pairs_found = False
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.columns[i] == 'class_encoded' or corr_matrix.columns[j] == 'class_encoded':
            continue
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            pairs_found = True
            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: r = {r:.4f}")
if not pairs_found:
    print("  None found! Features are relatively independent.")
    
print("\nMulticollinearity concern: LOW")
print("No feature pairs exceed |r| > 0.5, so we can use all features")
print("without worrying about redundancy destabilizing our models.")

**Correlation matrix decoded:**

The correlation matrix tells a fascinating story:

**With the target (class_encoded):** No single feature has a strong linear correlation with the target. The strongest correlations are in the 0.05-0.15 range, which means this is NOT a problem where one feature dominates. Our models will need to learn complex, possibly non-linear relationships between features to make good predictions. This is a genuinely hard classification problem.

**Between features:** The good news is that our features are largely independent of each other. No pair exceeds |r| = 0.5, which means we do not have a multicollinearity problem. Each feature brings unique information to the table. This is ideal for modeling -- we want features that each contribute something different rather than all saying the same thing in slightly different ways.

**Notable correlations:**
- `nswprice` and `vicprice` have some positive correlation (they are neighboring states, so their prices naturally move together)
- `nswdemand` and `vicdemand` similarly track each other weakly
- `transfer` has weak negative correlations with some features, suggesting interstate power flow partially compensates for local conditions

Key insight: This is a dataset where the magic will happen in feature interactions and non-linear patterns, not in single-feature linear relationships. Tree-based models should shine here.

### 4.4 Statistical Tests

Time for the formal evidence. We have been eyeballing differences -- now let's put numbers and p-values behind our hunches.

In [ ]:
print("STATISTICAL TESTS")
print("=" * 80)

# Mann-Whitney U tests for numerical features vs binary target
print("\nMann-Whitney U Test (numerical features vs target class)")
print("H0: Feature distributions are identical for UP and DOWN")
print("-" * 65)
print(f"{'Feature':<15} {'U-statistic':>15} {'p-value':>15} {'Significant?':>15}")
print("-" * 65)

for col in numerical_cols:
    down = df[df['class'] == 'DOWN'][col]
    up = df[df['class'] == 'UP'][col]
    u_stat, p_val = mannwhitneyu(down, up, alternative='two-sided')
    sig = "YES ***" if p_val < 0.001 else "YES **" if p_val < 0.01 else "YES *" if p_val < 0.05 else "NO"
    print(f"{col:<15} {u_stat:>15,.0f} {p_val:>15.2e} {sig:>15}")

In [ ]:
# Chi-square test for day vs class
print("\nChi-Square Test: Day of Week vs Price Direction")
print("H0: Day of week and price direction are independent")
print("-" * 55)

contingency = pd.crosstab(df['day'], df['class'])
chi2, p_val, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom:  {dof}")
print(f"p-value:             {p_val:.6f}")
print(f"Significant?         {'YES' if p_val < 0.05 else 'NO'} (alpha = 0.05)")

In [ ]:
# Normality tests
print("\nShapiro-Wilk Normality Test (on random 5000-sample subsets)")
print("H0: Data comes from a normal distribution")
print("-" * 55)

np.random.seed(42)
sample_idx = np.random.choice(len(df), size=5000, replace=False)

for col in numerical_cols:
    stat, p_val = shapiro(df[col].iloc[sample_idx])
    normal = "Normal" if p_val > 0.05 else "NOT Normal"
    print(f"  {col:<14} W={stat:.6f}  p={p_val:.2e}  --> {normal}")

print("\nVerdict: Nothing is normally distributed. Not even close.")
print("This confirms we should prefer non-parametric methods and tree-based models.")
print("Parametric assumptions are out the window -- and honestly, with")
print("electricity market data, that should surprise absolutely nobody.")

**Statistical test results -- the formal evidence:**

**Mann-Whitney U Tests:** Every single numerical feature shows a statistically significant difference between UP and DOWN classes (p < 0.001 for all). This means all features carry SOME information about the target. However, statistical significance is not the same as practical significance -- with 45,000+ samples, even tiny differences become "significant." The violin plots earlier gave us a better sense of which features have meaningful separation.

**Chi-Square Test:** The day-of-week vs class test tells us whether the day matters. If the p-value is significant, it means the day of the week has at least some relationship with price direction. Even if the effect is small (as we saw in the bar charts), the model might extract marginal value from it.

**Normality Tests:** Every feature fails the Shapiro-Wilk test spectacularly. Nothing in this dataset is normally distributed. This has two implications: (1) we should not trust any analysis that assumes normality, and (2) tree-based models (which make no distributional assumptions) are a natural fit for this data.

---
## 5. Feature Engineering

Now that we have interrogated every corner of this dataset, it is time to get creative. Feature engineering is where domain knowledge meets data intuition. We are going to create new features that capture patterns the raw data cannot express on its own.

In [ ]:
df_model = df.copy()

# 1. Price difference between states
df_model['price_diff'] = df_model['nswprice'] - df_model['vicprice']

# 2. Demand difference between states
df_model['demand_diff'] = df_model['nswdemand'] - df_model['vicdemand']

# 3. Price-to-demand ratio for each state (supply-demand pressure indicator)
df_model['nsw_price_demand_ratio'] = df_model['nswprice'] / (df_model['nswdemand'] + 1e-8)
df_model['vic_price_demand_ratio'] = df_model['vicprice'] / (df_model['vicdemand'] + 1e-8)

# 4. Total market price and demand
df_model['total_price'] = df_model['nswprice'] + df_model['vicprice']
df_model['total_demand'] = df_model['nswdemand'] + df_model['vicdemand']

# 5. Price volatility proxy (absolute difference from median)
df_model['nsw_price_deviation'] = abs(df_model['nswprice'] - df_model['nswprice'].median())
df_model['vic_price_deviation'] = abs(df_model['vicprice'] - df_model['vicprice'].median())

# 6. Transfer intensity (how aggressively power is being moved)
df_model['transfer_x_demand'] = df_model['transfer'] * df_model['total_demand']

# 7. Is it a weekend?
df_model['is_weekend'] = (df_model['day'].isin([6, 7])).astype(int)

# 8. Time-of-day bins (peak hours tend to have different price dynamics)
df_model['period_bin'] = pd.cut(df_model['period'], bins=4, labels=[0, 1, 2, 3]).astype(int)

# 9. Price spread (market tension indicator)
df_model['price_spread'] = abs(df_model['nswprice'] - df_model['vicprice'])

# 10. Demand-weighted transfer
df_model['demand_weighted_transfer'] = df_model['transfer'] / (df_model['total_demand'] + 1e-8)

new_features = [
    'price_diff', 'demand_diff', 'nsw_price_demand_ratio', 'vic_price_demand_ratio',
    'total_price', 'total_demand', 'nsw_price_deviation', 'vic_price_deviation',
    'transfer_x_demand', 'is_weekend', 'period_bin', 'price_spread', 
    'demand_weighted_transfer'
]

print(f"Created {len(new_features)} new features. Here is each with its target correlation:")
print("=" * 60)

for feat in new_features:
    corr = df_model[feat].corr(df_model['class_encoded'])
    bar = "|" * int(abs(corr) * 100)
    sign = "+" if corr > 0 else "-"
    print(f"  {feat:<30} {sign}{abs(corr):.4f}  {bar}")

print(f"\nTotal features now: {len(new_features) + len(numerical_cols) + 1} (original + engineered)")

**Feature engineering intuition -- why each feature exists:**

**price_diff & demand_diff**: Interstate asymmetry. When NSW is much more expensive than Victoria, or demand is lopsided, the market behaves differently. These capture the "tension" between the two states.

**price_demand_ratio**: The economics of it. High price relative to demand suggests supply constraints (price is high DESPITE moderate demand -- something is wrong on the supply side). Low ratio suggests oversupply.

**total_price & total_demand**: The overall market health. Sometimes what matters is not the individual states but the combined pressure on the whole system.

**price_deviation**: How abnormal is the current price? Deviations from the median capture whether we are in "normal" territory or in "everyone is panicking" territory.

**transfer_x_demand**: Transfer under high demand is very different from transfer under low demand. This interaction feature captures that nuance.

**is_weekend & period_bin**: Time features might be weak individually, but combined with other features they can capture patterns like "high price during peak hours on weekdays" that neither feature captures alone.

**price_spread**: The absolute gap between state prices. A wide spread means the market is stressed and arbitrage opportunities exist.

**demand_weighted_transfer**: How much transfer per unit of demand. This normalizes transfer by the market's overall activity level.

Now let's prepare everything for modeling.

In [ ]:
feature_cols = numerical_cols + new_features + ['day']

X = df_model[feature_cols].copy()
y = df_model['class_encoded'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution: {dict(y.value_counts())}")
print(f"  DOWN (0): {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"  UP   (1): {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")
print(f"\nStratification preserved:")
print(f"  Train UP%: {y_train.mean()*100:.1f}%  |  Test UP%: {y_test.mean()*100:.1f}%")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled. Data is ready for modeling.")

---
## 6. Modeling

The investigation has gathered all its evidence. Time for the trial. We will start with a simple baseline, then escalate to more powerful models, and see who can crack this case.

### 6.1 Baseline Model

In [ ]:
print("BASELINE: Logistic Regression")
print("=" * 50)
print("Why start here? Because if a linear model can solve it,")
print("we do not need the heavy artillery. Always establish your floor.\n")

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print(f"Accuracy:  {lr_acc:.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print(f"ROC-AUC:   {lr_auc:.4f}")
print(f"\nBaseline set at {lr_auc:.4f} AUC. Can we beat it?")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['DOWN', 'UP']))

Logistic regression gives us our floor. It is doing better than random (AUC > 0.5) but we are nowhere near satisfied. The linear model is trying to draw a straight line through what we know is a non-linear problem. Time to bring in the heavy hitters.

### 6.2 Model Selection

We will throw multiple algorithms at this problem and use 5-fold stratified cross-validation to get honest performance estimates. No peeking at the test set -- that stays locked in the vault until the very end.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42, learning_rate=0.1),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, random_state=42, use_label_encoder=False,
                                   eval_metric='logloss', scale_pos_weight=1.36),
    'LightGBM': lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1,
                                     scale_pos_weight=1.36),
    'KNN': KNeighborsClassifier(n_neighbors=11, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=42, algorithm='SAMME'),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 75)
print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'ROC-AUC':>10} {'Std':>10}")
print("-" * 75)

cv_results = {}

for name, model in models.items():
    if name in ['Logistic Regression', 'KNN']:
        acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        f1_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1', n_jobs=-1)
        auc_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    else:
        acc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
        auc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    cv_results[name] = {
        'accuracy': acc_scores.mean(),
        'f1': f1_scores.mean(),
        'auc': auc_scores.mean(),
        'auc_std': auc_scores.std()
    }
    
    print(f"{name:<25} {acc_scores.mean():>10.4f} {f1_scores.mean():>10.4f} {auc_scores.mean():>10.4f} {auc_scores.std():>10.4f}")

best_model_name = max(cv_results, key=lambda x: cv_results[x]['auc'])
print(f"\nBest model by ROC-AUC: {best_model_name} ({cv_results[best_model_name]['auc']:.4f})")

In [ ]:
# Visualization of model comparison
results_df = pd.DataFrame(cv_results).T.sort_values('auc', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(results_df)))
bars = ax.barh(results_df.index, results_df['auc'], color=colors,
               edgecolor='black', linewidth=0.5, height=0.6)

for bar, (idx, row) in zip(bars, results_df.iterrows()):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{row["auc"]:.4f}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('ROC-AUC Score', fontsize=12)
ax.set_title('Model Comparison: The Tournament', fontweight='bold', fontsize=14)
ax.set_xlim(0.5, max(results_df['auc']) + 0.06)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Random Guess')
ax.legend()

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**Model tournament results:**

The tree-based ensemble methods dominate, which is exactly what we predicted during the correlation analysis. When relationships are non-linear and features interact in complex ways, gradient boosting and random forests are in their element.

LightGBM and XGBoost are neck-and-neck at the top. These are both gradient boosting frameworks optimized for speed and performance, so their similar results make sense. Random Forest is close behind -- a strong showing from the old reliable.

Logistic Regression sits at the bottom, confirming that this problem needs non-linear modeling. KNN struggles because the high-dimensional feature space and mixed scales make distance-based methods less effective.

Let's take our top performer and see if we can squeeze more juice out of it.

### 6.3 Hyperparameter Tuning

In [ ]:
print("HYPERPARAMETER TUNING: LightGBM")
print("=" * 55)
print("Using GridSearchCV to find the optimal configuration.\n")

param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [4, 6, 8, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'min_child_samples': [20, 50],
    'subsample': [0.8, 1.0],
}

lgb_base = lgb.LGBMClassifier(
    random_state=42, verbose=-1, scale_pos_weight=1.36, n_jobs=-1
)

grid_search = GridSearchCV(
    lgb_base, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    refit=True
)

grid_search.fit(X_train, y_train)

print(f"Best ROC-AUC (CV): {grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

best_lgb = grid_search.best_estimator_

In [ ]:
# Also tune XGBoost for comparison
print("\nTuning XGBoost for comparison...")

xgb_param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 5],
}

xgb_base = xgb.XGBClassifier(
    random_state=42, use_label_encoder=False, 
    eval_metric='logloss', scale_pos_weight=1.36, n_jobs=-1
)

xgb_grid = GridSearchCV(
    xgb_base, xgb_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    refit=True
)

xgb_grid.fit(X_train, y_train)

print(f"Best XGBoost ROC-AUC (CV): {xgb_grid.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in xgb_grid.best_params_.items():
    print(f"  {param}: {value}")

best_xgb = xgb_grid.best_estimator_

print(f"\n--- TUNING SUMMARY ---")
print(f"LightGBM best CV AUC: {grid_search.best_score_:.4f}")
print(f"XGBoost  best CV AUC: {xgb_grid.best_score_:.4f}")

final_model = best_lgb if grid_search.best_score_ >= xgb_grid.best_score_ else best_xgb
final_model_name = "LightGBM" if grid_search.best_score_ >= xgb_grid.best_score_ else "XGBoost"
print(f"\nFinal model: {final_model_name}")

### 6.4 Model Evaluation

The model has been trained and tuned. Now comes the moment of truth -- we unleash it on the test set it has never seen. No second chances. No re-tuning. Just raw, honest performance.

In [ ]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_prob)
test_ap = average_precision_score(y_test, y_prob)

print(f"FINAL MODEL EVALUATION ({final_model_name})")
print("=" * 55)
print(f"Accuracy:           {test_acc:.4f}")
print(f"F1 Score:           {test_f1:.4f}")
print(f"ROC-AUC:            {test_auc:.4f}")
print(f"Average Precision:  {test_ap:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['DOWN', 'UP']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['DOWN', 'UP'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=13)

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#1565C0', linewidth=2.5, label=f'{final_model_name} (AUC={test_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Guess')
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#1565C0')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=11)

# Also add baseline LR for comparison
lr.fit(X_train_scaled, y_train)
y_prob_lr_test = lr.predict_proba(X_test_scaled)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr_test)
axes[1].plot(fpr_lr, tpr_lr, color='gray', linewidth=1.5, linestyle='--',
             label=f'Baseline LR (AUC={roc_auc_score(y_test, y_prob_lr_test):.4f})')
axes[1].legend(fontsize=10)

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(recall, precision, color='#FF5722', linewidth=2.5, label=f'AP={test_ap:.4f}')
axes[2].fill_between(recall, precision, alpha=0.15, color='#FF5722')
axes[2].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve', fontweight='bold', fontsize=13)
axes[2].legend(fontsize=11)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading the evaluation plots:**

**Confusion Matrix**: Look at the numbers. Most predictions land on the diagonal (correct predictions). The model makes errors in both directions -- some DOWN samples get predicted as UP and vice versa -- but the error rate is reasonable for both classes. It is not just predicting the majority class, which is exactly what we wanted to verify.

**ROC Curve**: The curve hugs the top-left corner nicely, staying well above the diagonal (random guess line). The area under the curve quantifies how well the model separates UP from DOWN across all threshold settings. The gap between our final model and the logistic regression baseline shows the value of non-linear modeling and feature engineering.

**Precision-Recall Curve**: This is especially important for our minority class (UP). The curve shows that even as we increase recall (catching more UP instances), precision does not collapse. The model maintains a reasonable tradeoff, which means it is not just throwing UP predictions at everything and hoping some stick.

In [ ]:
# Feature Importance
if hasattr(final_model, 'feature_importances_'):
    importances = pd.Series(final_model.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(importances)))
    importances.plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.3)
    
    ax.set_title(f'{final_model_name} Feature Importance: What Drove the Predictions?', 
                fontweight='bold', fontsize=14)
    ax.set_xlabel('Importance', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    print("=" * 45)
    for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).head(10).items(), 1):
        bar = "|" * int(imp / importances.max() * 30)
        print(f"  {i:>2}. {feat:<30} {imp:.4f}  {bar}")

**Feature importance revelations -- the final piece of the puzzle:**

The feature importance plot reveals which features the model leaned on most heavily. A few patterns emerge:

**Price features dominate.** This makes intuitive sense -- the current price is the strongest signal for where the price is going next. Momentum matters in energy markets.

**Engineered features earned their keep.** Several of our hand-crafted features (price_diff, demand ratios, deviation measures) rank among the top features. This validates our domain-driven feature engineering -- we did not just add noise, we added signal.

**Transfer matters.** Interstate power transfer carries real predictive value. The flow of electricity between NSW and Victoria is telling us something about the market's state that raw price and demand alone cannot capture.

**Time features are supporting actors.** Period and day are not the stars, but they contribute. The model uses them to modulate predictions based on when things happen, not just what is happening.

---
## 7. Conclusion

### 7.1 Summary of Findings

We came to this dataset with a simple question: can we predict whether electricity prices will go up or down? After 45,312 data points, 20+ features, 8 models, and more plots than a mystery novel, here is what we found:

**About the data:**
- The Australian electricity market is a fascinating beast. Prices are calm 95% of the time, then occasionally spike violently. This creates heavily skewed distributions that break every Gaussian assumption in the textbook.
- NSW and Victoria prices move somewhat together (they are interconnected states), but each has its own demand and pricing dynamics.
- Interstate power transfer is an underrated feature that captures market stress better than price or demand alone.
- Day of the week barely matters. Electrons do not know what a weekend is.

**About the features:**
- Price features are the strongest predictors. Current price momentum is the best signal for future direction.
- Engineered interaction features (price_diff, demand ratios, deviation from median) added meaningful predictive power beyond the raw data.
- No single feature dominates -- this is a genuinely multivariate problem that requires models capable of learning complex interactions.

### 7.2 Model Performance Verdict

In [ ]:
print("FINAL PERFORMANCE SUMMARY")
print("=" * 55)
print(f"\n  Baseline (Logistic Regression):")
print(f"    Accuracy:  {lr_acc:.4f}")
print(f"    ROC-AUC:   {lr_auc:.4f}")
print(f"\n  Final Model ({final_model_name}):")
print(f"    Accuracy:  {test_acc:.4f}")
print(f"    ROC-AUC:   {test_auc:.4f}")
print(f"    F1 Score:  {test_f1:.4f}")

improvement = ((test_auc - lr_auc) / lr_auc * 100)
print(f"\n  AUC Improvement over baseline: {improvement:+.1f}%")

if test_auc > 0.9:
    verdict = "Excellent. The model has strong discriminative power."
elif test_auc > 0.85:
    verdict = "Very good. The model captures meaningful patterns in the data."
elif test_auc > 0.8:
    verdict = "Good. There is clear learning, with room for further improvement."
elif test_auc > 0.7:
    verdict = "Decent. The model is learning but the problem is genuinely hard."
else:
    verdict = "Modest. This is a hard problem and the model captures only partial signal."

print(f"\n  Verdict: {verdict}")

### 7.3 Business and Real-World Implications

So what does this actually mean for someone operating in the Australian electricity market?

**If you are an energy trader:** This model gives you an edge. Even a modest improvement in predicting price direction translates to better buying and selling decisions. Over thousands of trades, that edge compounds into serious money. You would not bet your house on any single prediction, but as a probabilistic tool feeding into a larger trading system, this has real value.

**If you are a grid operator:** Knowing which direction prices are headed helps you pre-position resources. If the model says "UP" with high confidence, you can spin up reserve capacity or activate demand response programs before the spike hits instead of scrambling after.

**If you are a consumer (eventually):** Smart grids that can predict price movements will automatically shift your dishwasher and EV charging to cheap-price windows. You save money without thinking about it.

**What this model cannot do:** It cannot predict the magnitude of price changes, only the direction. It also operates on a snapshot basis -- it does not model temporal dependencies across consecutive periods. A future improvement would be to add lagged features (what happened in the last N periods) or use a sequence model like LSTM.

**The honest limitations:**
- This is normalized data -- we cannot reverse-engineer actual dollar predictions without the original scale
- No temporal modeling (each row is treated independently)
- The model was trained on historical data and market dynamics can shift

But as a starting point for understanding what drives electricity price movements in Australia, this notebook delivers. The grid never sleeps, and now we have a model that can read its mood.

---

**Find more work like this:**
- Kaggle: [kaggle.com/seki32](https://www.kaggle.com/seki32) -- Daily notebooks, experiments, and competitions
- GitHub: [github.com/Rekhii](https://github.com/Rekhii) -- All code and repositories

Thanks for reading. If you found this useful, an upvote costs you nothing and makes my day.